# Asteroid Cost Atlas — Data Explorer

Interactive notebook for querying the final ranked atlas (~1.5M asteroids, 33 columns).

**Prerequisite:** Run `make pipeline` before using this notebook.

In [ ]:
from pathlib import Path
from asteroid_cost_atlas.utils.query import CostAtlasDB

processed = Path("../data/processed")
for pattern in ("atlas_*.parquet", "sbdb_physical_*.parquet", "sbdb_orbital_*.parquet"):
    candidates = sorted(processed.glob(pattern))
    if candidates:
        break

if not candidates:
    raise FileNotFoundError("No processed parquet found. Run 'make pipeline' first.")

db = CostAtlasDB(candidates[-1])
print(f"Loaded: {candidates[-1].name}")
print(f"{len(db.sql('SELECT * FROM atlas LIMIT 0').columns)} columns")

In [ ]:
# Mission cost model parameters (used in all queries below)
LEO_COST_PER_KG = 2700      # $/kg to LEO (Falcon Heavy, 2024)
ISP = 320                    # seconds (bipropellant thruster)
G0 = 9.81                    # m/s²
VE = round(ISP * G0 / 1000, 4)  # exhaust velocity km/s

# SQL fragment for cost calculation — reused in every query
COST_SQL = f"{LEO_COST_PER_KG} * EXP(2 * delta_v_km_s / {VE})"
PROFIT_SQL = f"resource_value_usd_per_kg / ({COST_SQL})"

# Standard column set for all asteroid tables
STD_COLS = f"""
    economic_priority_rank AS rank,
    name,
    composition_class AS class,
    ROUND(diameter_estimated_km, 3) AS diameter_km,
    diameter_source AS d_source,
    ROUND(delta_v_km_s, 2) AS delta_v_km_s,
    ROUND(estimated_value_usd, 0) AS value_usd,
    ROUND({COST_SQL}, 0) AS cost_per_kg_usd,
    resource_value_usd_per_kg AS resource_per_kg_usd,
    ROUND({PROFIT_SQL}, 3) AS profit_ratio,
    ROUND(moid_au, 4) AS moid_au,
    neo,
    orbit_class
"""

print(f"Cost model: LEO=${LEO_COST_PER_KG}/kg, Isp={ISP}s, Ve={VE} km/s")
print(f"Round-trip cost examples: dv=1→${LEO_COST_PER_KG * 2.718**(2*1/VE):,.0f}/kg, "
      f"dv=5→${LEO_COST_PER_KG * 2.718**(2*5/VE):,.0f}/kg, "
      f"dv=10→${LEO_COST_PER_KG * 2.718**(2*10/VE):,.0f}/kg")

---
## Reference: Column & Value Definitions

### Standard columns (present in all asteroid tables below)

| Column | Unit | Description |
|---|---|---|
| `rank` | — | Economic priority rank (1 = best target). Based on `value × accessibility` |
| `name` | — | Official asteroid designation |
| `class` | — | Composition class: **C** (carbonaceous), **S** (silicaceous), **M** (metallic), **V** (basaltic), **U** (unknown) |
| `diameter_km` | km | Best available diameter estimate |
| `d_source` | — | Diameter provenance: **measured** (telescope/radar) or **estimated** (from H magnitude) |
| `delta_v_km_s` | km/s | Mission delta-v proxy — energy cost to reach the asteroid. Lower = cheaper |
| `value_usd` | USD | Estimated total extractable resource value of the asteroid |
| `cost_per_kg_usd` | USD/kg | Estimated round-trip delivery cost per kg (Tsiolkovsky + Falcon Heavy baseline) |
| `resource_per_kg_usd` | USD/kg | Resource value per kg of raw asteroid material |
| `profit_ratio` | — | `resource_per_kg / cost_per_kg`. **> 1 = theoretically profitable** |
| `moid_au` | AU | Minimum orbit intersection distance to Earth. Lower = closer approach |
| `neo` | Y/N | Near-Earth Object flag |
| `orbit_class` | — | Orbital classification (see below) |

### Composition classes

| Class | Full name | Albedo | Primary resources | Value/kg | % of catalog |
|---|---|---|---|---|---|
| **C** | Carbonaceous | 0.03–0.10 | Water (10–20%), organics, carbon | $500 | 5.7% |
| **S** | Silicaceous | 0.15–0.35 | Silicates, iron/nickel, trace PGMs | $1 | 3.6% |
| **M** | Metallic | 0.10–0.25 | Iron/nickel (80%), PGMs (~50 ppm) | $50 | 0.1% |
| **V** | Basaltic | 0.30–0.50 | Pyroxene, feldspar — low value | $0.10 | 0.4% |
| **U** | Unknown | — | No taxonomy or albedo — uses population average | $10 | 90.2% |

### Orbit classes

| Code | Name | Description |
|---|---|---|
| **APO** | Apollo | NEO, a > 1 AU, crosses Earth orbit (outward) |
| **ATE** | Aten | NEO, a < 1 AU, crosses Earth orbit (inward) |
| **AMO** | Amor | NEO, 1.017 < perihelion < 1.3 AU, approaches but doesn't cross Earth |
| **IEO** | Atira | NEO, orbit entirely inside Earth's |
| **MBA** | Main Belt | Between Mars and Jupiter (2.0–3.3 AU) — vast majority of catalog |
| **IMB** | Inner Main Belt | Inner edge of main belt |
| **OMB** | Outer Main Belt | Outer edge of main belt |
| **MCA** | Mars Crosser | Crosses Mars orbit |
| **TJN** | Jupiter Trojan | Shares Jupiter's orbit (L4/L5 points) |
| **TNO** | Trans-Neptunian | Beyond Neptune — Pluto, Eris, etc. |
| **CEN** | Centaur | Between Jupiter and Neptune |

### Diameter source

| Value | Method | Confidence |
|---|---|---|
| **measured** | Telescope, radar, or spacecraft observation (SBDB/NEOWISE) | High |
| **estimated** | Derived from absolute magnitude H via `D = 1329/√pV × 10^(-H/5)` | Moderate (±50% typical) |

### NEO / PHA flags

| Flag | Meaning |
|---|---|
| **neo = Y** | Perihelion < 1.3 AU — approaches Earth's neighborhood |
| **pha = Y** | NEO with MOID < 0.05 AU and H < 22 — large enough and close enough to be hazardous |

### Cost model

| Parameter | Value | Source |
|---|---|---|
| LEO launch cost | $2,700/kg | Falcon Heavy (2024 pricing) |
| Specific impulse | 320 s | Bipropellant (MMH/NTO) |
| Round-trip factor | 2× | Simplification: outbound + return |
| Formula | `cost = LEO_cost × exp(2 × dv / Ve)` | Tsiolkovsky rocket equation |

`profit_ratio > 1` means the asteroid's resource value per kg exceeds the delivery cost — **theoretically profitable**.

---
## 1. Catalog Overview

In [ ]:
db.sql("""
    SELECT
        COUNT(*) AS total_asteroids,
        COUNT(economic_score) AS economically_scored,
        COUNTIF(neo = 'Y') AS near_earth_objects,
        COUNTIF(pha = 'Y') AS potentially_hazardous,
        COUNT(diameter_km) AS measured_diameters,
        COUNT(diameter_estimated_km) AS total_diameters,
        COUNT(rotation_hours) AS has_rotation,
        COUNT(taxonomy) AS has_taxonomy
    FROM atlas
""")

In [ ]:
db.stats()

---
## 2. Top 20 Economic Targets (all asteroids)

In [ ]:
db.sql(f"""
    SELECT {STD_COLS}
    FROM atlas
    WHERE economic_priority_rank IS NOT NULL
    ORDER BY economic_priority_rank
    LIMIT 20
""")

---
## 3. Top 20 NEO Mining Targets

In [ ]:
db.sql(f"""
    SELECT {STD_COLS}
    FROM atlas
    WHERE neo = 'Y' AND economic_score IS NOT NULL
    ORDER BY economic_score DESC
    LIMIT 20
""")

---
## 4. Profitable NEOs (profit_ratio > 1)

Asteroids where the resource value per kg exceeds the estimated delivery cost. Sorted by total value.

In [ ]:
db.sql(f"""
    SELECT {STD_COLS}
    FROM atlas
    WHERE neo = 'Y'
      AND economic_score IS NOT NULL
      AND ({PROFIT_SQL}) > 1.0
    ORDER BY estimated_value_usd DESC
    LIMIT 30
""")

---
## 5. Most Accessible AND Viable (NEOs > 100m, delta-v < 6 km/s)

In [ ]:
db.sql(f"""
    SELECT {STD_COLS}
    FROM atlas
    WHERE neo = 'Y'
      AND diameter_estimated_km > 0.1
      AND delta_v_km_s < 6.0
    ORDER BY delta_v_km_s
    LIMIT 30
""")

---
## 6. Large NEOs with Low Delta-v (> 500m, < 7 km/s)

In [ ]:
db.sql(f"""
    SELECT {STD_COLS}
    FROM atlas
    WHERE neo = 'Y'
      AND diameter_estimated_km > 0.5
      AND delta_v_km_s < 7.0
    ORDER BY economic_score DESC
    LIMIT 20
""")

---
## 7. Metallic (M-type) Targets — Platinum Group Metals

In [ ]:
db.sql(f"""
    SELECT {STD_COLS}
    FROM atlas
    WHERE composition_class = 'M'
    ORDER BY economic_score DESC
    LIMIT 20
""")

---
## 8. Physical Feasibility: Best Mining Candidates

Asteroids with rotation + regolith data AND high feasibility scores.

In [ ]:
db.sql(f"""
    SELECT {STD_COLS},
           ROUND(surface_gravity_m_s2, 6) AS gravity_m_s2,
           ROUND(rotation_feasibility, 3) AS rot_feasibility,
           ROUND(regolith_likelihood, 3) AS regolith
    FROM atlas
    WHERE rotation_feasibility IS NOT NULL
      AND regolith_likelihood > 0.5
      AND rotation_feasibility > 0.5
    ORDER BY economic_score DESC
    LIMIT 20
""")

---
## 9. Composition Class Breakdown

In [ ]:
db.sql(f"""
    SELECT composition_class AS class, composition_source AS source,
           COUNT(*) AS count,
           resource_value_usd_per_kg AS value_per_kg,
           ROUND(AVG(diameter_estimated_km), 2) AS avg_diameter_km,
           ROUND(AVG(delta_v_km_s), 2) AS avg_delta_v,
           ROUND(AVG({COST_SQL}), 0) AS avg_cost_per_kg,
           ROUND(AVG(estimated_value_usd), 0) AS avg_value_usd
    FROM atlas
    WHERE economic_score IS NOT NULL
    GROUP BY composition_class, composition_source, resource_value_usd_per_kg
    ORDER BY count DESC
""")

---
## 10. Economic Value Distribution by Composition

In [ ]:
db.sql(f"""
    SELECT composition_class AS class,
           COUNT(*) AS count,
           resource_value_usd_per_kg AS value_per_kg,
           ROUND(SUM(estimated_value_usd), 0) AS total_value_usd,
           ROUND(AVG(estimated_value_usd), 0) AS avg_value_usd,
           ROUND(MEDIAN(estimated_value_usd), 0) AS median_value_usd,
           ROUND(AVG({COST_SQL}), 0) AS avg_cost_per_kg,
           ROUND(AVG({PROFIT_SQL}), 3) AS avg_profit_ratio
    FROM atlas
    WHERE economic_score IS NOT NULL
    GROUP BY composition_class, resource_value_usd_per_kg
    ORDER BY total_value_usd DESC
""")

---
## 11. Orbit Class Distribution

In [ ]:
db.sql(f"""
    SELECT orbit_class,
           COUNT(*) AS count,
           ROUND(AVG(delta_v_km_s), 2) AS avg_delta_v,
           ROUND(AVG(diameter_estimated_km), 2) AS avg_diameter_km,
           ROUND(AVG(estimated_value_usd), 0) AS avg_value_usd,
           ROUND(AVG({COST_SQL}), 0) AS avg_cost_per_kg,
           COUNTIF(composition_class = 'C') AS c_type,
           COUNTIF(composition_class = 'S') AS s_type,
           COUNTIF(composition_class = 'M') AS m_type
    FROM atlas
    WHERE orbit_class IS NOT NULL
    GROUP BY orbit_class
    ORDER BY count DESC
    LIMIT 15
""")

---
## 12. Delta-v Distribution

In [ ]:
db.delta_v_histogram(bin_width=1.0)

---
## 13. Diameter Source & Quality

In [ ]:
db.sql("""
    SELECT diameter_source,
           COUNT(*) AS count,
           ROUND(MIN(diameter_estimated_km), 4) AS min_km,
           ROUND(MEDIAN(diameter_estimated_km), 4) AS median_km,
           ROUND(MAX(diameter_estimated_km), 2) AS max_km
    FROM atlas
    WHERE diameter_source IS NOT NULL
    GROUP BY diameter_source
    ORDER BY count DESC
""")

---
## 14. Taxonomy Distribution (from LCDB)

In [ ]:
db.sql(f"""
    SELECT taxonomy,
           COUNT(*) AS count,
           ROUND(AVG(albedo), 3) AS avg_albedo,
           ROUND(AVG(diameter_estimated_km), 2) AS avg_diameter_km,
           ROUND(AVG(delta_v_km_s), 2) AS avg_delta_v,
           ROUND(AVG(estimated_value_usd), 0) AS avg_value_usd,
           ROUND(AVG({COST_SQL}), 0) AS avg_cost_per_kg
    FROM atlas
    WHERE taxonomy IS NOT NULL
    GROUP BY taxonomy
    ORDER BY count DESC
    LIMIT 15
""")

---
## 15. Custom Query

Edit the SQL below. The view is called `atlas` (33 columns).
Use `STD_COLS` for the standard column set, or write your own SELECT.

In [ ]:
db.sql(f"""
    SELECT {STD_COLS}
    FROM atlas
    LIMIT 10
""")

In [ ]:
db.close()